#Sample Shock Analysis Based on a SAM Table

In [ ]:
import numpy as np
import pandas as pd

Step 1. Define the SAM matrix.

In [ ]:
# Adjusted 5x5 SAM data with zeros on the main diagonal
sam_matrix = np.array([
    [0, 50, 30, 60, 20],
    [40, 0, 70, 90, 30],
    [30, 50, 0, 80, 40],
    [60, 90, 80, 0, 60],
    [20, 30, 40, 60, 0]
])
# Create a Pandas DataFrame
columns = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']
sam_df = pd.DataFrame(sam_matrix, columns=columns, index=columns)
display(sam_df)

In [ ]:
from pyspark.sql.types import IntegerType, StructType, StructField

# Column names
columns = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']

# Convert the NumPy array to a list of tuples and cast the values to int
sam_list = [tuple(int(value) for value in row) for row in sam_matrix]

# Create the schema for the Spark DataFrame
schema = StructType([StructField(column, IntegerType(), nullable=True) for column in columns])

# Create a Spark DataFrame directly from the list of tuples and the schema
sam_spark_df = spark.createDataFrame(sam_list, schema=schema)

# Display the Spark DataFrame
display(sam_spark_df)

Step 2. Convert the SAM matrix into a Databricks dataframe. Use the long format.

In [ ]:
# Convert to long format (similar to Pandas melt)
from pyspark.sql.functions import col, lit, array, struct, explode

# Convert wide format to long format
long_format_df = sam_spark_df.select(
    array([lit(c).alias("Column") for c in columns]).alias("Column"),
    struct([col(c).alias(c) for c in columns]).alias("Values")
).select(explode(col("Column")).alias("Column"), col("Values.*")).withColumnRenamed("Column", "Row")

long_format_df = long_format_df.selectExpr(
    "Row",
    "stack(5, 'Agriculture', Agriculture, 'Manufacturing', Manufacturing, 'Services', Services, 'Households', Households, 'Government', Government) as (Column, Value)"
)

# Display the long format Spark DataFrame
display(long_format_df)

Step 3. Normalize and regularize the SAM: The SAM is normalized to get the technical coefficients matrix and then regularized to ensure numerical stability and handle potential singularities. 

In [ ]:
# Normalize the SAM matrix by dividing each element by the sum of its column
column_sums = sam_matrix.sum(axis=0)
normalized_sam = sam_matrix / column_sums

# Regularization parameter
epsilon = 1e-10

# Add regularization to the diagonal - the np.eye() function generates an identity matrix
regularized_matrix = normalized_sam + np.eye(normalized_sam.shape[0]) * epsilon

Step 4. Compute the Leontief Inverse: The Leontief inverse is computed with regularization to handle potential singularity.

In [ ]:
# Compute the inverse of the normalized SAM matrix
try:
    inverse_sam = np.linalg.inv(regularized_matrix)
except np.linalg.LinAlgError as e:
    # The matrix is not invertible. Compute the Moore-Penrose pseudoinverse instead. 
    inverse_sam = np.linalg.pinv(regularized_matrix)

Step 5. Perform Shock Analysis: A 10% increase in final demand for the Agriculture sector is introduced, and the impact on all sectors is calculated using the Leontief inverse.

In [ ]:
# Perform Shock Analysis: 10% increase in final demand for Agriculture
shock_vector = np.array([0.10, 0, 0, 0, 0])

# Calculate the impact of the shock
impact_vector = inverse_sam @ shock_vector
print(impact_vector)

Convert the impact vector to a dataframe for display.

In [ ]:
# Convert the impact_vector to a Spark DataFrame
sectors = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']
impact_df = spark.createDataFrame([(sectors[i], float(impact_vector[i])) for i in range(len(sectors))], ["Sector", "Impact"])

# Display the Spark DataFrame
display(impact_df)

Step 1. Create the SAM DataFrame: The SAM is created with zeros on the main diagonal and loaded into a Pandas dataframe. The Pandas dataframe format is the easiest format to represent the SAM matrix as a table.


In [ ]:
import pandas as pd

# Adjusted 5x5 SAM data with zeros on the main diagonal
data = np.array([
    [0, 50, 30, 60, 20],
    [40, 0, 70, 90, 30],
    [30, 50, 0, 80, 40],
    [60, 90, 80, 0, 60],
    [20, 30, 40, 60, 0]
])
# Create a Pandas DataFrame
columns = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']
sam_df = pd.DataFrame(data, columns=columns, index=columns)
display(sam_df)

Step 2. Convert the SAM matrix into a Databricks dataframe.

In [ ]:
from pyspark.sql.types import IntegerType, StructType, StructField

# Column names
columns = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']

# Convert the NumPy array to a list of tuples and cast the values to int
sam_list = [tuple(int(value) for value in row) for row in data]

# Create the schema for the Spark DataFrame
schema = StructType([StructField(column, IntegerType(), nullable=True) for column in columns])

# Create a Spark DataFrame directly from the list of tuples and the schema
sam_spark_df = spark.createDataFrame(sam_list, schema=schema)

# Display the Spark DataFrame
display(sam_spark_df)

Convert to Long Format: The Spark DataFrame is transformed to a long format using Spark SQL functions. This involves selecting columns, creating an array of column names, exploding the array to create rows, and stacking the values.

The long format offers several advantages in data analysis, manipulation, and visualization. It is more flexible, easier to work with, and compatible with many data processing and visualization tools

In [ ]:
# Convert to long format (similar to Pandas melt)
from pyspark.sql.functions import col, lit, array, struct, explode

# Convert wide format to long format
long_format_df = sam_spark_df.select(
    array([lit(c).alias("Column") for c in columns]).alias("Column"),
    struct([col(c).alias(c) for c in columns]).alias("Values")
).select(explode(col("Column")).alias("Column"), col("Values.*")).withColumnRenamed("Column", "Row")

long_format_df = long_format_df.selectExpr(
    "Row",
    "stack(5, 'Agriculture', Agriculture, 'Manufacturing', Manufacturing, 'Services', Services, 'Households', Households, 'Government', Government) as (Column, Value)"
)

# Display the long format Spark DataFrame
display(long_format_df)

Step 3. Normalize the SAM: The SAM is normalized to get the technical coefficients matrix (A).

In [ ]:
# Normalize the SAM to obtain technical coefficients matrix (A)
# Convert to numpy array for normalization
sam_array = data

# Normalize by column sums to get technical coefficients matrix (A)
column_sums = sam_array.sum(axis=0)
A = sam_array / column_sums

Step 4. Convert Normalized SAM to Spark DataFrame: The normalized SAM is converted back to a Spark DataFrame and displayed.

In [ ]:
# Convert back to Spark DataFrame -- with a temporary conversion into pandas dataframe
A_df = spark.createDataFrame(pd.DataFrame(A, columns=columns, index=columns).reset_index().melt(id_vars=['index'], var_name='Column', value_name='Value'))
A_df = A_df.withColumnRenamed('index', 'Row')
display(A_df)

Step 5. Compute the Leontief Inverse: The Leontief inverse is computed with regularization to handle potential singularity.

In [ ]:
# Compute the Leontief inverse
regularized_matrix = A + np.eye(normalized_sam.shape[0]) * epsilon
# np.eye creates an identity matrix of the same size as A (using normalized_sam as the reference for dimensions) and multiplies it by ?.

# compute the inverse of the square matrix
try:
    leontief_inverse = np.linalg.inv(regularized_matrix)
except np.linalg.LinAlgError:
    leontief_inverse = np.linalg.pinv(regularized_matrix)


Step 6. Convert Leontief Inverse to Spark DataFrame: The Leontief inverse is converted to a Spark DataFrame and displayed.

In [ ]:
leontief_inverse_df = spark.createDataFrame(pd.DataFrame(leontief_inverse, columns=columns, index=columns).reset_index().melt(id_vars=['index'], var_name='Column', value_name='Value'))
leontief_inverse_df = leontief_inverse_df.withColumnRenamed('index', 'Row')
display(leontief_inverse_df)

Step 7. Calculate Multipliers: The multipliers are calculated by summing the columns of the Leontief inverse, converted to Dataframe and displayed.

In [ ]:
# Calculate multipliers
multipliers = leontief_inverse.sum(axis=0)
multipliers_df = spark.createDataFrame(pd.DataFrame(multipliers, columns=["Multiplier"], index=columns).reset_index().rename(columns={'index': 'Sector'}))
display(multipliers_df)

Step 8. Perform Shock Analysis: A 10% increase in final demand for the Agriculture sector is introduced, and the impact on all sectors is calculated using the Leontief inverse.

In [ ]:
# Perform Shock Analysis: 10% increase in final demand for Agriculture
shock_vector = np.zeros(5)
shock_vector[0] = 0.10  # 10% increase in Agriculture

# Calculate the impact of the shock
impact_vector = leontief_inverse @ shock_vector
impact_df = spark.createDataFrame(pd.DataFrame(impact_vector, columns=["Impact"], index=columns).reset_index().rename(columns={'index': 'Sector'}))
display(impact_df)


## Incorporate the impact analysis results back into the initial Social Accounting Matrix (SAM) using the RAS method.

In [ ]:
"""
Reciprocal Alteration of Selection

The RAS method iteratively adjusts the rows and columns of the original SAM to match the new target sums, ensuring the matrix remains balanced.

Adjusts the rows and columns of the matrix to match the target sums iteratively.

The adjustment is done by scaling the rows and columns using the factors R and S.
"""

def ras_method(A, target_row_sums, target_col_sums, max_iter=100, tol=1e-5):
    """
    Perform the RAS method to update the SAM matrix to match target row and column sums.
    """
    n = A.shape[0]
    R = np.ones(n)
    S = np.ones(n)
    for iteration in range(max_iter):
        # Adjust rows
        for i in range(n):
            R[i] = target_row_sums[i] / np.sum(A[i, :] * S)
        # Adjust columns
        for j in range(n):
            S[j] = target_col_sums[j] / np.sum(A[:, j] * R)
        # Update A
        A_new = (A.T * R).T * S
        # Check convergence
        if np.allclose(np.sum(A_new, axis=1), target_row_sums, atol=tol) and np.allclose(np.sum(A_new, axis=0), target_col_sums, atol=tol):
            break
        A = A_new
    return A

# Calculate the target row and column sums after impact analysis
original_row_sums = sam_matrix.sum(axis=1)
original_col_sums = sam_matrix.sum(axis=0)
target_row_sums = original_row_sums + impact_vector
target_col_sums = original_col_sums + impact_vector

# Apply the RAS method to update the SAM matrix
updated_sam = ras_method(sam_matrix, target_row_sums, target_col_sums)

print("Updated SAM matrix:")
print(updated_sam)

Create a dataframe of the updated SAM matrix into for pretty display.

In [ ]:
from pyspark.sql.types import StructType, StructField, FloatType

# Convert the updated_sam matrix to a list of tuples
updated_sam_list = [tuple(map(float, row)) for row in updated_sam]
print(updated_sam_list)

# Define column names
columns = ['Agriculture', 'Manufacturing', 'Services', 'Households', 'Government']

# Define schema 
schema = StructType([StructField(col, FloatType(), True) for col in columns])

# Create a Spark DataFrame
updated_sam_df = spark.createDataFrame(updated_sam_list, schema=schema)

# Display the Spark DataFrame
display(updated_sam_df)

## Perform the same impact analysis calculation in R

In [ ]:
%r

# Define the SAM matrix
sam_matrix <- matrix(c(0, 50, 30, 60, 20,
                       40, 0, 70, 90, 30,
                       30, 50, 0, 80, 40,
                       60, 90, 80, 0, 60,
                       20, 30, 40, 60, 0), 
                     nrow = 5, byrow = TRUE)

colnames(sam_matrix) <- c('Agriculture', 'Manufacturing', 'Services', 'Households', 'Government')
rownames(sam_matrix) <- c('Agriculture', 'Manufacturing', 'Services', 'Households', 'Government')

# Normalize the SAM matrix by dividing each element by the sum of its column
column_sums <- colSums(sam_matrix)
normalized_sam <- sam_matrix / column_sums[col(sam_matrix)]

# Regularization parameter
epsilon <- 1e-10

# Add regularization to the diagonal
regularized_matrix <- normalized_sam + diag(epsilon, nrow(normalized_sam), ncol(normalized_sam))

# Function to compute the inverse and handle non-invertible matrices
invert_matrix <- function(matrix) {
  tryCatch({
    inv_matrix <- solve(matrix)
    return(inv_matrix)
  }, error = function(e) {
    cat("Matrix is not invertible:", e$message, "\n")
    return(NULL)
  })
}

# Compute the inverse of the regularized SAM matrix
inverse_sam <- invert_matrix(regularized_matrix)

if (!is.null(inverse_sam)) {
  cat("Normalized SAM matrix:\n")
  print(normalized_sam)
  cat("\nRegularized SAM matrix:\n")
  print(regularized_matrix)
  cat("\nInverse matrix of the regularized SAM:\n")
  print(inverse_sam)
  
  # Perform Shock Analysis: 10% increase in final demand for Agriculture
  shock_vector <- c(0.10, 0, 0, 0, 0)
  
  # Calculate the impact of the shock
  impact_vector <- inverse_sam %*% shock_vector
  
  # Convert the impact_vector to a Spark DataFrame
  sectors <- c('Agriculture', 'Manufacturing', 'Services', 'Households', 'Government')
  impact_df <- data.frame(Sector = sectors, Impact = as.numeric(impact_vector))
#  impact_df <- as.DataFrame(data.frame(Sector = sectors, Impact = as.numeric(impact_vector)))
  
  # Display the Spark DataFrame
  display(impact_df)
}

